In [ ]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

# ================================
# CONFIGURAÇÕES
# ================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESAS = ["PDV ITZ 01", "PDV ITZ 02"]  # Adicione aqui suas empresas

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_itz;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# ================================
# FUNÇÕES
# ================================
def testar_token():
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESAS[0],  # Testa a primeira empresa da lista
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido!")
        return True
    else:
        print(f"❌ Token inválido ({resp.status_code})")
        return False

def coletar_pedidos_intervalo(start: datetime, end: datetime, empresa: str):
    """Coleta pedidos de um intervalo de tempo específico com paginação"""
    df_total = pd.DataFrame()
    pagina = 1
    limite = 1000

    while True:
        params = {
            "dataInicial": start.strftime("%Y-%m-%dT%H:%M:%S"),
            "dataFinal": end.strftime("%Y-%m-%dT%H:%M:%S"),
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": empresa,
            "pagina": pagina,
            "limite": limite
        }
        resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=180)
        if resp.status_code != 200:
            print(f"⚠️ Erro {resp.status_code} em {start} -> {end} para empresa {empresa}")
            break

        dados = resp.json()
        if isinstance(dados, dict):
            df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
        elif isinstance(dados, list):
            df = pd.DataFrame(dados)
        else:
            df = pd.DataFrame()

        if df.empty:
            break

        col_complex = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (dict,list))).any()]
        for col in col_complex:
            df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)

        df_total = pd.concat([df_total, df], ignore_index=True)
        if len(df) < limite:
            break
        pagina += 1
        time.sleep(0.1)

    return df_total

def coletar_pedidos_dia(dia: datetime, empresa: str):
    """Divide o dia em intervalos de 1 hora"""
    df_dia = pd.DataFrame()
    hora_inicio = datetime(dia.year, dia.month, dia.day, 0, 0, 0)
    while hora_inicio < datetime(dia.year, dia.month, dia.day, 23, 59, 59):
        hora_fim = min(hora_inicio + timedelta(hours=1), datetime(dia.year, dia.month, dia.day, 23, 59, 59))
        df_intervalo = coletar_pedidos_intervalo(hora_inicio, hora_fim, empresa)
        if not df_intervalo.empty:
            df_dia = pd.concat([df_dia, df_intervalo], ignore_index=True)
        hora_inicio = hora_fim + timedelta(seconds=1)

    if "ID" in df_dia.columns:
        df_dia = df_dia.drop_duplicates(subset=["ID"])
    print(f"📦 Total de registros no dia {dia.strftime('%Y-%m-%d')} para empresa {empresa}: {len(df_dia)}")
    return df_dia

def coletar_ano_multithread(ano: int, empresa: str, max_workers: int = 5):
    """Coleta todo o ano usando múltiplas threads"""
    dias = [datetime(ano, 1, 1) + timedelta(days=i) for i in range(0, 365)]
    df_ano_total = pd.DataFrame()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(coletar_pedidos_dia, dia, empresa): dia for dia in dias}
        for future in as_completed(futures):
            df_dia = future.result()
            if not df_dia.empty:
                df_ano_total = pd.concat([df_ano_total, df_dia], ignore_index=True)

    if "ID" in df_ano_total.columns:
        df_ano_total = df_ano_total.drop_duplicates(subset=["ID"])
    return df_ano_total

def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    df = df.loc[:, ~df.columns.duplicated()]  # Remove colunas duplicadas pelo nome (SEGURANÇA)
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print(f"💾 Gravados {len(df)} registros na tabela {tabela}!")

# ================================
# EXECUÇÃO
# ================================
if __name__ == "__main__":
    ANO = 2025
    MAX_THREADS = 5  # ajuste conforme sua máquina

    if not testar_token():
        print("❌ Corrija o token antes de continuar")
    else:
        with engine.begin() as conn:
            conn.execute(text("DROP TABLE IF EXISTS pedidos"))

        # Coleta dados de todas as empresas individualmente e concatena resultados como um 'filter'
        df_total = pd.DataFrame()
        for empresa in EMPRESAS:
            print(f"\nIniciando coleta para empresa: {empresa}")
            df_ano = coletar_ano_multithread(ANO, empresa, max_workers=MAX_THREADS)
            if not df_ano.empty:
                # Preenche/atualiza a coluna empresa
                df_ano['empresa'] = empresa
                df_ano = df_ano.loc[:, ~df_ano.columns.duplicated()]  # Elimina duplicadas
                df_total = pd.concat([df_total, df_ano], ignore_index=True)
            else:
                print(f"⚠️ Nenhum dado coletado para a empresa {empresa}")

        # Depois da coleta de todas, salva tudo junto, filtrado, como um único DataFrame!
        armazenar_no_sql(df_total, tabela="pedidos")
        print("🏁 Processo finalizado!")


✅ Token válido!

Iniciando coleta para empresa: PDV ITZ 01
📦 Total de registros no dia 2025-01-01 para empresa PDV ITZ 01: 0
📦 Total de registros no dia 2025-01-05 para empresa PDV ITZ 01: 2
📦 Total de registros no dia 2025-01-02 para empresa PDV ITZ 01: 62
📦 Total de registros no dia 2025-01-03 para empresa PDV ITZ 01: 78
📦 Total de registros no dia 2025-01-04 para empresa PDV ITZ 01: 80
📦 Total de registros no dia 2025-01-06 para empresa PDV ITZ 01: 80
📦 Total de registros no dia 2025-01-07 para empresa PDV ITZ 01: 82
📦 Total de registros no dia 2025-01-08 para empresa PDV ITZ 01: 70
📦 Total de registros no dia 2025-01-09 para empresa PDV ITZ 01: 59
📦 Total de registros no dia 2025-01-10 para empresa PDV ITZ 01: 47
📦 Total de registros no dia 2025-01-12 para empresa PDV ITZ 01: 56
📦 Total de registros no dia 2025-01-11 para empresa PDV ITZ 01: 92
📦 Total de registros no dia 2025-01-14 para empresa PDV ITZ 01: 38
📦 Total de registros no dia 2025-01-13 para empresa PDV ITZ 01: 60
📦 Tot

In [1]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed

# ================================
# CONFIGURAÇÕES
# ================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESAS = ["PDV ITZ 01", "PDV ITZ 02", "PDV ITZ 03", "PDV ITZ 04"]  # Adicione aqui suas empresas

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_itz;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# ================================
# FUNÇÕES
# ================================
def testar_token():
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESAS[0],  # Testa a primeira empresa da lista
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido!")
        return True
    else:
        print(f"❌ Token inválido ({resp.status_code})")
        return False

def coletar_pedidos_intervalo(start: datetime, end: datetime, empresa: str):
    """Coleta pedidos de um intervalo de tempo específico com paginação"""
    df_total = pd.DataFrame()
    pagina = 1
    limite = 1000

    while True:
        params = {
            "dataInicial": start.strftime("%Y-%m-%dT%H:%M:%S"),
            "dataFinal": end.strftime("%Y-%m-%dT%H:%M:%S"),
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": empresa,
            "pagina": pagina,
            "limite": limite
        }
        resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=180)
        if resp.status_code != 200:
            print(f"⚠️ Erro {resp.status_code} em {start} -> {end} para empresa {empresa}")
            break

        dados = resp.json()
        if isinstance(dados, dict):
            df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
        elif isinstance(dados, list):
            df = pd.DataFrame(dados)
        else:
            df = pd.DataFrame()

        if df.empty:
            break

        col_complex = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (dict,list))).any()]
        for col in col_complex:
            df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)

        df_total = pd.concat([df_total, df], ignore_index=True)
        if len(df) < limite:
            break
        pagina += 1
        time.sleep(0.1)

    return df_total

def coletar_pedidos_dia(dia: datetime, empresa: str):
    """Divide o dia em intervalos de 1 hora"""
    df_dia = pd.DataFrame()
    hora_inicio = datetime(dia.year, dia.month, dia.day, 0, 0, 0)
    while hora_inicio < datetime(dia.year, dia.month, dia.day, 23, 59, 59):
        hora_fim = min(hora_inicio + timedelta(hours=1), datetime(dia.year, dia.month, dia.day, 23, 59, 59))
        df_intervalo = coletar_pedidos_intervalo(hora_inicio, hora_fim, empresa)
        if not df_intervalo.empty:
            df_dia = pd.concat([df_dia, df_intervalo], ignore_index=True)
        hora_inicio = hora_fim + timedelta(seconds=1)

    if "ID" in df_dia.columns:
        df_dia = df_dia.drop_duplicates(subset=["ID"])
    print(f"📦 Total de registros no dia {dia.strftime('%Y-%m-%d')} para empresa {empresa}: {len(df_dia)}")
    return df_dia

def coletar_ano_multithread(ano: int, empresa: str, max_workers: int = 5):
    """Coleta todo o ano usando múltiplas threads"""
    dias = [datetime(ano, 1, 1) + timedelta(days=i) for i in range(0, 365)]
    df_ano_total = pd.DataFrame()

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(coletar_pedidos_dia, dia, empresa): dia for dia in dias}
        for future in as_completed(futures):
            df_dia = future.result()
            if not df_dia.empty:
                df_ano_total = pd.concat([df_ano_total, df_dia], ignore_index=True)

    if "ID" in df_ano_total.columns:
        df_ano_total = df_ano_total.drop_duplicates(subset=["ID"])
    return df_ano_total

def armazenar_no_sql(df: pd.DataFrame, tabela: str):
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    df = df.loc[:, ~df.columns.duplicated()]  # Remove colunas duplicadas pelo nome (SEGURANÇA)
    df.to_sql(tabela, con=engine, if_exists="append", index=False, chunksize=500)
    print(f"💾 Gravados {len(df)} registros na tabela {tabela}!")

# ================================
# EXECUÇÃO
# ================================
if __name__ == "__main__":
    ANO = 2025
    MAX_THREADS = 5  # ajuste conforme sua máquina

    if not testar_token():
        print("❌ Corrija o token antes de continuar")
    else:
        with engine.begin() as conn:
            conn.execute(text("DROP TABLE IF EXISTS pedidos"))

        df_total = pd.DataFrame()
        for empresa in EMPRESAS:
            print(f"\nIniciando coleta para empresa: {empresa}")
            df_ano = coletar_ano_multithread(ANO, empresa, max_workers=MAX_THREADS)
            if not df_ano.empty:
                # Sempre use a coluna 'Empresa' da API e defina seu valor
                df_ano['Empresa'] = empresa
                # Remove colunas duplicadas
                df_ano = df_ano.loc[:, ~df_ano.columns.duplicated()]
                df_total = pd.concat([df_total, df_ano], ignore_index=True)
            else:
                print(f"⚠️ Nenhum dado coletado para a empresa {empresa}")

        armazenar_no_sql(df_total, tabela="pedidos")
        print("🏁 Processo finalizado!")


✅ Token válido!

Iniciando coleta para empresa: PDV ITZ 01
📦 Total de registros no dia 2025-01-05 para empresa PDV ITZ 01: 2
📦 Total de registros no dia 2025-01-01 para empresa PDV ITZ 01: 0
📦 Total de registros no dia 2025-01-02 para empresa PDV ITZ 01: 62
📦 Total de registros no dia 2025-01-03 para empresa PDV ITZ 01: 78
📦 Total de registros no dia 2025-01-04 para empresa PDV ITZ 01: 80
📦 Total de registros no dia 2025-01-07 para empresa PDV ITZ 01: 82
📦 Total de registros no dia 2025-01-06 para empresa PDV ITZ 01: 80
📦 Total de registros no dia 2025-01-08 para empresa PDV ITZ 01: 70
📦 Total de registros no dia 2025-01-09 para empresa PDV ITZ 01: 59
📦 Total de registros no dia 2025-01-10 para empresa PDV ITZ 01: 47
📦 Total de registros no dia 2025-01-12 para empresa PDV ITZ 01: 56
📦 Total de registros no dia 2025-01-11 para empresa PDV ITZ 01: 92📦 Total de registros no dia 2025-01-14 para empresa PDV ITZ 01: 38

📦 Total de registros no dia 2025-01-15 para empresa PDV ITZ 01: 51
📦 Tot

In [1]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text
import urllib
import time
import json
from datetime import datetime, timedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
from calendar import monthrange

# ================================
# CONFIGURAÇÕES
# ================================
API_BASE = "https://api.sigecloud.com.br/request/Pedidos/Pesquisar"
EMPRESAS = ["PDV ITZ 01", "PDV ITZ 02", "PDV ITZ 03", "PDV ITZ 04"]

HEADERS = {
    "Authorization-Token": "4aac320039947465f02b71a95bc13a8ef55a465ef99e035ecf83aa8323152dd4eff9a2463da8d2206d1005982ab67f9e9f17edc62fd02199a2488b029952c247c6db2098e33fa37bfb85b75d1adec8077fa46c7b81b193513faf2dd8e7edd985d6a2f2b03a202b5a07a104735ae280701d33da1f039975a0d122140386e73d95",
    "User": "Diego.oalbuquerque@hotmail.com",
    "App": "API"
}

# SQL Server
params_engine = urllib.parse.quote_plus(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    "SERVER=localhost;"
    "DATABASE=sige_itz;"
    "Trusted_Connection=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params_engine}")

# ================================
# FUNÇÕES
# ================================
def testar_token():
    params = {
        "dataInicial": "2025-01-01",
        "filtrarPor": "DataFaturamentoPedido",
        "empresa": EMPRESAS[0],
        "pagina": 1,
        "limite": 1
    }
    resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=30)
    if resp.status_code == 200:
        print("✅ Token válido!")
        return True
    else:
        print(f"❌ Token inválido ({resp.status_code})")
        return False

def criar_tabela_se_nao_existe():
    """Cria a tabela pedidos se ela não existir"""
    query_criar = """
    IF NOT EXISTS (SELECT * FROM sys.tables WHERE name = 'pedidos')
    BEGIN
        CREATE TABLE pedidos (
            ID NVARCHAR(255) PRIMARY KEY,
            Empresa NVARCHAR(255),
            -- Adicione aqui outras colunas conforme necessário
            -- Exemplo: DataFaturamento DATETIME, ValorTotal DECIMAL(18,2), etc.
        )
    END
    """
    with engine.begin() as conn:
        conn.execute(text(query_criar))
    print("✅ Tabela 'pedidos' verificada/criada")

def coletar_pedidos_intervalo(start: datetime, end: datetime, empresa: str):
    """Coleta pedidos de um intervalo de tempo específico com paginação"""
    df_total = pd.DataFrame()
    pagina = 1
    limite = 1000

    while True:
        params = {
            "dataInicial": start.strftime("%Y-%m-%dT%H:%M:%S"),
            "dataFinal": end.strftime("%Y-%m-%dT%H:%M:%S"),
            "filtrarPor": "DataFaturamentoPedido",
            "empresa": empresa,
            "pagina": pagina,
            "limite": limite
        }
        resp = requests.get(API_BASE, headers=HEADERS, params=params, timeout=180)
        if resp.status_code != 200:
            print(f"⚠️ Erro {resp.status_code} em {start} -> {end} para empresa {empresa}")
            break

        dados = resp.json()
        if isinstance(dados, dict):
            df = pd.DataFrame(dados.get("data") or dados.get("dados") or dados.get("result") or [])
        elif isinstance(dados, list):
            df = pd.DataFrame(dados)
        else:
            df = pd.DataFrame()

        if df.empty:
            break

        col_complex = [col for col in df.columns if df[col].apply(lambda x: isinstance(x, (dict,list))).any()]
        for col in col_complex:
            df[col] = df[col].apply(lambda x: json.dumps(x) if x is not None else None)

        df_total = pd.concat([df_total, df], ignore_index=True)
        if len(df) < limite:
            break
        pagina += 1
        time.sleep(0.1)

    return df_total

def coletar_pedidos_dia(dia: datetime, empresa: str):
    """Divide o dia em intervalos de 1 hora"""
    df_dia = pd.DataFrame()
    hora_inicio = datetime(dia.year, dia.month, dia.day, 0, 0, 0)
    while hora_inicio < datetime(dia.year, dia.month, dia.day, 23, 59, 59):
        hora_fim = min(hora_inicio + timedelta(hours=1), datetime(dia.year, dia.month, dia.day, 23, 59, 59))
        df_intervalo = coletar_pedidos_intervalo(hora_inicio, hora_fim, empresa)
        if not df_intervalo.empty:
            df_dia = pd.concat([df_dia, df_intervalo], ignore_index=True)
        hora_inicio = hora_fim + timedelta(seconds=1)

    if "ID" in df_dia.columns:
        df_dia = df_dia.drop_duplicates(subset=["ID"])
    print(f"📦 Total de registros no dia {dia.strftime('%Y-%m-%d')} para empresa {empresa}: {len(df_dia)}")
    return df_dia

def coletar_mes_atual_multithread(ano: int, mes: int, empresa: str, max_workers: int = 5):
    """Coleta apenas o mês especificado usando múltiplas threads"""
    ultimo_dia = monthrange(ano, mes)[1]
    dias = [datetime(ano, mes, dia) for dia in range(1, ultimo_dia + 1)]
    df_mes_total = pd.DataFrame()

    print(f"📅 Coletando dados de {len(dias)} dias do mês {mes}/{ano} para empresa {empresa}")

    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {executor.submit(coletar_pedidos_dia, dia, empresa): dia for dia in dias}
        for future in as_completed(futures):
            df_dia = future.result()
            if not df_dia.empty:
                df_mes_total = pd.concat([df_mes_total, df_dia], ignore_index=True)

    if "ID" in df_mes_total.columns:
        df_mes_total = df_mes_total.drop_duplicates(subset=["ID"])
    return df_mes_total

def armazenar_no_sql_incremental(df: pd.DataFrame, tabela: str):
    """Armazena dados evitando duplicatas usando tabela temporária + MERGE"""
    if df.empty:
        print("⚠️ Nenhum dado para gravar")
        return
    
    df = df.loc[:, ~df.columns.duplicated()]
    
    # 1. Criar tabela temporária
    tabela_temp = f"{tabela}_temp"
    df.to_sql(tabela_temp, con=engine, if_exists="replace", index=False, chunksize=500)
    print(f"📥 Dados temporários carregados ({len(df)} registros)")
    
    # 2. Fazer MERGE (inserir apenas novos registros)
    query_merge = f"""
    MERGE INTO {tabela} AS target
    USING {tabela_temp} AS source
    ON target.ID = source.ID
    WHEN NOT MATCHED THEN
        INSERT ({', '.join(df.columns)})
        VALUES ({', '.join(['source.' + col for col in df.columns])})
    WHEN MATCHED THEN
        UPDATE SET {', '.join([f'target.{col} = source.{col}' for col in df.columns if col != 'ID'])};
    """
    
    with engine.begin() as conn:
        result = conn.execute(text(query_merge))
        conn.execute(text(f"DROP TABLE {tabela_temp}"))
    
    print(f"💾 Dados sincronizados na tabela {tabela} (novos + atualizados)!")

# ================================
# EXECUÇÃO - MÊS ATUAL
# ================================
if __name__ == "__main__":
    # Pega o mês e ano atuais automaticamente
    hoje = datetime.now()
    ANO_ATUAL = hoje.year
    MES_ATUAL = hoje.month
    MAX_THREADS = 5

    print(f"🚀 Iniciando coleta do mês atual: {MES_ATUAL}/{ANO_ATUAL}")

    if not testar_token():
        print("❌ Corrija o token antes de continuar")
    else:
        criar_tabela_se_nao_existe()  # Garante que a tabela existe
        
        df_total = pd.DataFrame()
        for empresa in EMPRESAS:
            print(f"\n🏢 Coletando dados para empresa: {empresa}")
            df_mes = coletar_mes_atual_multithread(ANO_ATUAL, MES_ATUAL, empresa, max_workers=MAX_THREADS)
            
            if not df_mes.empty:
                df_mes['Empresa'] = empresa
                df_mes = df_mes.loc[:, ~df_mes.columns.duplicated()]
                df_total = pd.concat([df_total, df_mes], ignore_index=True)
            else:
                print(f"⚠️ Nenhum dado coletado para a empresa {empresa}")

        armazenar_no_sql_incremental(df_total, tabela="pedidos")
        print(f"\n🏁 Processo finalizado! Total de registros coletados: {len(df_total)}")

🚀 Iniciando coleta do mês atual: 3/2026
✅ Token válido!
✅ Tabela 'pedidos' verificada/criada

🏢 Coletando dados para empresa: PDV ITZ 01
📅 Coletando dados de 31 dias do mês 3/2026 para empresa PDV ITZ 01
📦 Total de registros no dia 2026-03-05 para empresa PDV ITZ 01: 71
📦 Total de registros no dia 2026-03-03 para empresa PDV ITZ 01: 73
📦 Total de registros no dia 2026-03-02 para empresa PDV ITZ 01: 80
📦 Total de registros no dia 2026-03-01 para empresa PDV ITZ 01: 75
📦 Total de registros no dia 2026-03-04 para empresa PDV ITZ 01: 70
📦 Total de registros no dia 2026-03-08 para empresa PDV ITZ 01: 74
📦 Total de registros no dia 2026-03-06 para empresa PDV ITZ 01: 91
📦 Total de registros no dia 2026-03-07 para empresa PDV ITZ 01: 114
📦 Total de registros no dia 2026-03-09 para empresa PDV ITZ 01: 104
📦 Total de registros no dia 2026-03-10 para empresa PDV ITZ 01: 71
📦 Total de registros no dia 2026-03-11 para empresa PDV ITZ 01: 72
📦 Total de registros no dia 2026-03-12 para empresa PDV I